# 🇵🇭 Hiligaynon ↔ English Dictionary — Data Loading & Index Building

This notebook:
1. Loads and merges both JSON dictionary datasets
2. Inspects the data (counts, samples)
3. Creates train/dev evaluation examples
4. Builds the ChromaDB vector index with vowel-normalized metadata
5. Tests vowel-aware retrieval (o↔u, i↔e)

In [8]:
import sys, os, json
from pathlib import Path

# Project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print(f"Project root: {ROOT}")
print(f"Files: {[f.name for f in ROOT.iterdir() if f.is_file()]}")

Project root: c:\Users\gioan\Documents\GitHub\chatbutt
Files: ['.env', '.gitattributes', '.gitignore', 'dictionary.json', 'english-hiligaynon-akeanon.html', 'hiligaynon_dictionary.json', 'hiligaynon_words.pdf', 'README.md', 'requirements.lock.txt', 'requirements.txt']


## 1. Load and Merge Both Datasets

In [9]:
from src.load_data import merge_datasets, save_merged, create_examples

# Merge both dictionary files
entries = merge_datasets(
    pinoy_path=ROOT / "hiligaynon_dictionary.json",
    concise_path=ROOT / "dictionary.json",
)

print(f"Total merged entries: {len(entries):,}")
print(f"  From pinoydictionary: {sum(1 for e in entries if e['source'] == 'pinoydictionary'):,}")
print(f"  From concise dict:    {sum(1 for e in entries if e['source'] == 'concise'):,}")
print()

# Show sample entries
for e in entries[:3]:
    print(f"  [{e['source']}] {e['word']}: {e['definition'][:100]}...")
print("  ...")
for e in entries[-3:]:
    print(f"  [{e['source']}] {e['word']}: {e['definition'][:100]}...")

Total merged entries: 23,656
  From pinoydictionary: 23,556
  From concise dict:    100

  [pinoydictionary] a: A suffix of verbs that have a passive in-on. This suffix occurs in the following tenses: 1.) The pas...
  [pinoydictionary] a: Ah, Oh, Well, Why. A, amó gid inâ. Ah, that is it, certainly. A, ikáw galî ang nagabút. Oh, it is yo...
  [pinoydictionary] a: The letter A in Visayan is pronounced as in Spanish, except when it has a cut short, abrupt sound, w...
  ...
  [concise] atip: large board or piece of wood used as a brace or support, particularly when grinding...
  [concise] atipuyong: feel faint, be dizzy...
  [concise] atis: a fruit, the sugar apple: Anona squamosa Linn....


## 2. Save Merged Dictionary & Create Train/Dev Examples

In [10]:
# Save merged dictionary
save_merged(entries, ROOT / "data" / "processed" / "dictionary.json")

# Create train/dev splits
train, dev = create_examples(entries, n_train=60, n_dev=30, out_dir=ROOT / "data" / "examples")

print("\nSample train example:")
print(json.dumps(train[0], indent=2, ensure_ascii=False))

Saved 23656 merged entries → c:\Users\gioan\Documents\GitHub\chatbutt\data\processed\dictionary.json
Created 60 train + 30 dev examples → c:\Users\gioan\Documents\GitHub\chatbutt\data\examples

Sample train example:
{
  "hiligaynon": "asawa",
  "english": "spouse: husband, wife; married (said of a woman)",
  "definition": "spouse: husband, wife; married (said of a woman)"
}


## 3. Build ChromaDB Vector Index

In [11]:
from src.retriever import build_index

# Build the ChromaDB index from the merged dictionary
collection = build_index(
    dictionary_path=ROOT / "data" / "processed" / "dictionary.json",
    persist_dir=ROOT / "chroma_db"
)

print(f"Collection: {collection.name}")
print(f"Total indexed entries: {collection.count()}")

Indexed 23656 entries into ChromaDB at c:\Users\gioan\Documents\GitHub\chatbutt\chroma_db
Collection: hiligaynon_dictionary
Total indexed entries: 23656


## 4. Test Vowel-Aware Retrieval

Test that the retriever correctly finds entries even when vowels are swapped (o↔u, i↔e).

In [12]:
from src.retriever import retrieve_vowel_aware, retrieve_for_sentence, normalize_vowels, generate_vowel_variants

# Test vowel variant generation
test_words = ["buot", "boot", "maayo", "diin", "deen", "tubig"]
for w in test_words:
    variants = generate_vowel_variants(w)
    normalized = normalize_vowels(w)
    print(f"{w:10s} → normalized: {normalized:10s} | variants: {variants}")

print("\n" + "="*60)

# Test vowel-aware retrieval
queries = ["buot", "maayo", "deen", "balay", "tubig"]
for q in queries:
    results = retrieve_vowel_aware(q, collection, top_k=3)
    print(f"\n🔍 Query: '{q}'")
    for doc in results[:3]:
        print(f"   → {doc[:100]}...")

print("\n" + "="*60)

# Test sentence-level retrieval
sentence = "Maayo nga aga, diin ka na subong?"
print(f"\n📝 Sentence retrieval: '{sentence}'")
results = retrieve_for_sentence(sentence, collection, top_k_per_word=3)
for doc in results[:5]:
    print(f"   → {doc[:100]}...")

buot       → normalized: buut       | variants: ['boot', 'buut', 'bout', 'buot']
boot       → normalized: buut       | variants: ['boot', 'buut', 'bout', 'buot']
maayo      → normalized: maayu      | variants: ['maayo', 'maayu']
diin       → normalized: diin       | variants: ['diin', 'dein', 'deen', 'dien']
deen       → normalized: diin       | variants: ['diin', 'dein', 'deen', 'dien']
tubig      → normalized: tubig      | variants: ['tubeg', 'tobig', 'tubig', 'tobeg']


🔍 Query: 'buot'
   → Hiligaynon: sapín | Definition: (Sp. zapin) Shoe, boot. (see sapátos, bótas, butítos)....
   → Hiligaynon: bótas | Definition: (Sp. bota) Boot, shoe, footwear. (see sapátos, sapín, butítos)....
   → Hiligaynon: panikód-tíkod | Definition: To stamp one's heels on the ground, push or kick with one's ...

🔍 Query: 'maayo'
   → Hiligaynon: maayó-áyo | Definition: Beautiful, handsome, good-looking, pretty, goodly, comely, pleas...
   → Hiligaynon: magayón | Definition: Beautiful, handsome, nice, etc. 